<a href="https://colab.research.google.com/github/starlton/Deep-Learning/blob/main/Week%202/week2_micrograd_extended.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — micrograd: A Tiny Autograd Engine (Extended)

**Inspiration:** - Andrej Karpathy Zero to Hero course on YouTube.

**Goal:** Build an automatic differentiation engine from scratch — the same core idea that powers PyTorch and TensorFlow.

Autograd is what lets a neural network compute the gradient of its loss with respect to every parameter *automatically*, no matter how complex the expression. We build a minimal version that operates on scalar values.

**The big idea:** every math expression is a **computational graph**. Each node holds a value. To train, we need the gradient of the output with respect to every input — obtained by walking the graph *backward*, applying the **chain rule** at each node.

**What this notebook covers:**
1. The `Value` class — wraps a number, tracks how it was computed
2. Core operations: `+`, `*` — build the graph, store local gradient rules
3. Extended operations: `**`, negation, subtraction, `tanh`, `relu`
4. `backward()` — propagates gradients through the whole graph
5. **Gradient checking** — proving every rule is correct against a numerical estimate

---

## Setup — Imports

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt

%matplotlib inline

---
## The Theory — Computational Graphs and the Chain Rule

Running example:

$$f(a, b, c) = (a + b) \cdot c$$

Broken into single operations:

$$u = a + b \qquad v = c \cdot u \qquad (v = f)$$

As a computational graph:

```
a ──┐
    ├──> (+) ──> u ──┐
b ──┘               ├──> (*) ──> v
c ──────────────────┘
```

### Forward pass (a=2, b=3, c=4)
$$u = 2 + 3 = 5 \qquad v = 4 \cdot 5 = 20$$

### Backward pass — the chain rule
Each operation has a **local gradient**:

**Addition** ($u = a + b$) — gradient passes through unchanged:
$$\frac{\partial u}{\partial a} = 1 \qquad \frac{\partial u}{\partial b} = 1$$

**Multiplication** ($v = c \cdot u$) — each input's gradient is the *other* input's value:
$$\frac{\partial v}{\partial c} = u = 5 \qquad \frac{\partial v}{\partial u} = c = 4$$

**Chaining to the leaves** via $\frac{\partial v}{\partial a} = \frac{\partial v}{\partial u}\cdot\frac{\partial u}{\partial a}$:

$$\frac{\partial v}{\partial a} = 4 \qquad \frac{\partial v}{\partial b} = 4 \qquad \frac{\partial v}{\partial c} = 5$$

Target output: **a.grad=4, b.grad=4, c.grad=5**.

---

## The Universal Pattern

Every operation's `_backward` follows the **same shape**:

$$\texttt{input.grad}\; \mathrel{+}= \; (\text{local derivative}) \times \texttt{out.grad}$$

- **local derivative** = how the output changes w.r.t. that input (pure calculus for that one operation)
- **out.grad** = the gradient flowing in from downstream (the chain rule)
- **`+=`** not `=` — a value used in multiple places accumulates gradient from every path (multivariable chain rule)

Once you see this pattern, adding *any* operation is just: figure out its local derivative, multiply by `out.grad`.

| Operation | Local derivative w.r.t. input | `_backward` line |
|---|---|---|
| `a + b` | 1 | `self.grad += out.grad` |
| `a * b` | the other operand | `self.grad += other.data * out.grad` |
| `a ** n` | $n \cdot a^{n-1}$ | `self.grad += n * self.data**(n-1) * out.grad` |
| `tanh(a)` | $1 - \tanh^2(a)$ | `self.grad += (1 - out.data**2) * out.grad` |
| `relu(a)` | 1 if a>0 else 0 | `self.grad += (1 if self.data>0 else 0) * out.grad` |

## The `Value` Class

| Attribute | Purpose |
|---|---|
| `data` | The actual number |
| `grad` | Gradient of the final output w.r.t. this value (starts at 0) |
| `_children` | The Values that produced this one (graph edges) |
| `_backward` | Function that pushes this node's gradient to its children |

### Notes on the extended operations

**Constant handling:** `__add__` and `__mul__` wrap raw numbers in `Value` so that `Value(3) * 2` works, not just `Value(3) * Value(2)`.

**Power (`__pow__`):** the exponent stays a plain number (a fixed constant, not a trainable value), so it's not a child and gets no gradient. Only the base does.

**Negation & subtraction:** built by reusing existing ops — `-x` is `x * -1`, and `a - b` is `a + (-b)` — so their gradients come for free.

**tanh:** its derivative $1 - \tanh^2(x)$ is computable entirely from the output (`out.data` is already $\tanh(x)$).

**relu:** gradient passes through (×1) when the input is positive, blocked (×0) otherwise.

In [2]:
class Value:
  def __init__(self, data, _children=()): #grad, children, operation):
    self.data = data
    self.grad = 0.0 ## Condition
    self._children = set(_children)
    self._backward = lambda: None

  def __repr__(self):
    return f"Value(data={self.data})"


  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other) # Fix bug so we can use raw values => Value(1.0) + 1 works.

    out = Value(self.data + other.data, _children=(self, other))
    def _backward():
      self.grad +=  out.grad
      other.grad += out.grad

    out._backward = _backward
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other) # Fix bug so we can use raw values => Value(1.0) * 1 works.

    out = Value(self.data * other.data, _children=(self, other))

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad

    out._backward = _backward
    return out



  def backward(self):
    # 1. Topological order all of the children in the graph
    topo = []
    visited = set()
    def build_topo(node):
      if node not in visited:
        visited.add(node)
        for child in node._children:
          build_topo(child)

        topo.append(node)

    build_topo(self)

    # 2. Seed the output gradient
    self.grad = 1.0

    # 3. Apply the chain rule in reverse topological order
    for node in reversed(topo):
      node._backward()



  # DAY 2 Adding functions

  def __pow__(self, other):
      assert isinstance(other, (int, float)), "only supporting int/float powers"

      out = Value(self.data ** other, _children=(self,))

      # Should be power rule for computing gradient
      def _backward():
        self.grad += other * (self.data**(other-1)) * out.grad
        #other.grad += self.data * (other**(self.data-1)) * out.grad  ------------------ NO reason to compute others gradient, normally a fixed val anyways.

      out._backward = _backward
      return out

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def tanh(self):
    out = Value(math.tanh(self.data), _children=(self,))

    def _backward():
      self.grad += (1 - out.data**2) * out.grad

    out._backward = _backward
    return out

  def relu(self):
    out = Value(max(0, self.data), _children=(self,))

    def _backward():
      self.grad += (1 if self.data > 0 else 0) * out.grad

    out._backward = _backward
    return out

### Walking through `backward()`

**Why topological sort?** Gradient can only flow into a node once everything downstream of it has computed its gradient. Topological order guarantees that when reversed, we always process a node before the nodes that feed into it.

**Why seed with 1.0?** $\frac{\partial v}{\partial v} = 1$ — the gradient of the output w.r.t. itself. This is the spark that starts the chain.

**Why reversed order?** We start at the output (grad = 1) and push gradient backward toward the inputs, one operation at a time.

---

## Test 1 — The Core Example

$$f(a,b,c) = (a + b) \cdot c, \quad a=2,\ b=3,\ c=4$$

Expected: `f.data` = 20, and `(a.grad, b.grad, c.grad, f.grad)` = `(4, 4, 5, 1)`.

In [3]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

f = (a+b) * c


f.backward()

a.grad, b.grad, c.grad, f.grad

(4.0, 4.0, 5.0, 1.0)

**Result `(4.0, 4.0, 5.0, 1.0)`** — matches the hand derivation exactly. The engine builds the graph automatically and computes all gradients in one backward pass.

---

## Test 2 — Power and Composition

Testing the new `__pow__` operation: $f = a^3 + c$.

The gradient w.r.t. $a$ should be $\frac{d}{da}a^3 = 3a^2 = 3 \cdot 2^2 = 12$.

In [4]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

f = (a**3) + c
f.data

12.0

In [5]:
f.backward()
a.grad, b.grad, c.grad, f.grad

(12.0, 0.0, 1.0, 1.0)

Note: `b.grad` is 0 here because `b` doesn't appear in the expression $a^3 + c$ — nudging `b` doesn't change `f` at all. `a.grad` is 12 (the power rule), and `c.grad` is 1 (addition passes through).

---

## Test 3 — tanh Activation

Testing a composition with a nonlinearity: $c = a \cdot \tanh(b)$.

In [6]:
a = Value(2.0)
b = Value(3.0)

c = a * b.tanh()

print(c)

c.backward()

a.grad, b.grad

Value(data=1.990109507373461)


(0.9950547536867305, 0.019732074330880423)

---
## Gradient Checking — Proving Every Rule Is Correct

This is the payoff. We verify each operation's **analytical** gradient (from `.backward()`) against a **numerical** estimate using the symmetric difference from Week 3 Day 1:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

If two completely independent methods agree, the `_backward` rule is correct — not on faith, but on proof.

The helper takes two versions of the same function: the micrograd version (operates on a `Value`) and the plain-math version (operates on a float).

In [7]:
def grad_check(micrograd_fn, plain_fn, x_val, h=1e-6):
    """
    Compares micrograd's analytical gradient against the numerical gradient.

    micrograd_fn: takes a Value, returns a Value  (uses .backward())
    plain_fn:     takes a float, returns a float  (plain math)
    x_val:        the point to check the gradient at
    """
    # 1. Analytical gradient (micrograd)
    x = Value(x_val)
    out = micrograd_fn(x)
    out.backward()
    analytical = x.grad

    # 2. Numerical gradient (symmetric difference)
    numerical = (plain_fn(x_val + h) - plain_fn(x_val - h)) / (2 * h)

    match = np.isclose(analytical, numerical, atol=1e-4)
    status = "PASS" if match else "FAIL"
    print(f"[{status}] at x={x_val}:  analytical={analytical:.6f}  numerical={numerical:.6f}")

In [8]:
# Check each operation: (micrograd version, plain math version)
print("Power (x**2):")
grad_check(lambda x: x**2,        lambda t: t**2,         3.0)

print("\nNegation (-x):")
grad_check(lambda x: -x,          lambda t: -t,           4.0)

print("\nSubtraction (x - 2):")
grad_check(lambda x: x - 2,       lambda t: t - 2,        7.0)

print("\ntanh:")
grad_check(lambda x: x.tanh(),    lambda t: math.tanh(t), 0.5)

print("\nReLU (positive input):")
grad_check(lambda x: x.relu(),    lambda t: max(0, t),    3.0)

print("\nReLU (negative input):")
grad_check(lambda x: x.relu(),    lambda t: max(0, t),   -2.0)

Power (x**2):
[PASS] at x=3.0:  analytical=6.000000  numerical=6.000000

Negation (-x):
[PASS] at x=4.0:  analytical=-1.000000  numerical=-1.000000

Subtraction (x - 2):
[PASS] at x=7.0:  analytical=1.000000  numerical=1.000000

tanh:
[PASS] at x=0.5:  analytical=0.786448  numerical=0.786448

ReLU (positive input):
[PASS] at x=3.0:  analytical=1.000000  numerical=1.000000

ReLU (negative input):
[PASS] at x=-2.0:  analytical=0.000000  numerical=0.000000


### What to look for

Every line should print **PASS**. The most interesting is **tanh at x=0.5**: analytical should be $1 - \tanh^2(0.5) \approx 0.7864$, and the numerical estimate lands right on top of it.

When all six pass, you've *proven* your backprop rules correct by two independent methods — the deepest kind of confidence there is.

---

## Summary — What We Built

| Component | What it does |
|---|---|
| `Value` | Wraps a scalar, tracks graph + gradient |
| `+`, `*` | Core ops with constant handling |
| `**`, neg, `-` | Extended algebra, reusing core ops |
| `tanh`, `relu` | Nonlinear activations + their derivatives |
| `backward()` | Topological sort + reverse chain rule |
| `grad_check` | Numerical verification of every rule |

**The one idea behind it all:** every `_backward` is `(local derivative) × out.grad`. That's the chain rule, mechanized.

**Next: build `Neuron`, `Layer`, and `MLP` on top of `Value`, then train on XOR.**